# Baselines comparison

Run BayesBreak alongside the upstream-driven baseline wrappers in
`bayesbreak.baselines`. We do **not** re-implement these algorithms —
each call is a thin wrapper around the canonical package.

This notebook covers the pure-Python wrappers (`ruptures` + the
`fearnhead_exact` reference). R-backed wrappers (`cbs`, `smuce`,
`rjmcmc`) are listed at the end with install hints.

In [ ]:
import numpy as np
from bayesbreak import BayesBreakGaussian
from bayesbreak.baselines import segment_with, available_algorithms

print('available:', available_algorithms())

rng = np.random.default_rng(0)
n = 200
true_b = [70, 130]
y = np.r_[
    rng.normal(0.0, 0.3, 70),
    rng.normal(2.0, 0.3, 60),
    rng.normal(-1.0, 0.3, 70),
]
X = np.arange(n).reshape(-1, 1)

## BayesBreak

In [ ]:
bb = BayesBreakGaussian(k_max=8).fit(X, y)
print(f'BayesBreak: k_map={bb.k_map_}, boundaries={bb.map_boundaries_[1:-1]}')

## Ruptures (PELT, Optimal Partitioning, BS, WBS)

In [ ]:
for name, kwargs in [
    ('pelt',                 dict(penalty=10.0)),
    ('optimal_partitioning', dict(n_bkps=2)),
    ('binary_segmentation',  dict(n_bkps=2)),
    ('wild_binary_segmentation',
                             dict(n_bkps=2, random_state=0, n_random_windows=30)),
]:
    res = segment_with(name, y, **kwargs)
    print(f'{name:>26s}: k={res.k}, boundaries={res.boundaries.tolist()}')

## Fearnhead-exact-DP reference
Drives BayesBreak's own DP at the Fearnhead-2006 prior choice
(geometric `p(k)`, optional length-aware cohesion). Labelled
reference comparator — no standalone third-party implementation.

In [ ]:
res = segment_with('fearnhead_exact', y, k_max=8, geometric_rate=0.3)
print(f'fearnhead_exact: k={res.k}, boundaries={res.boundaries.tolist()}')
print(f'  provenance: {res.package} v{res.package_version}')
print(f'  extra: {res.extra}')

## R-backed baselines (install hints)

These require `pip install bayesbreak[baselines-r]` plus the R packages.
See [Installation](../installation.md):

- **CBS** (Olshen et al. 2004) via `DNAcopy::segment`:
  ```python
  res = segment_with('cbs', y_log2ratio, alpha=0.01, nperm=10_000)
  ```
- **SMUCE** (Frick, Munk & Sieling 2014) via `stepR::stepFit`:
  ```python
  res = segment_with('smuce', y, alpha=0.05, family='gauss')
  ```
- **RJMCMC-style MCMC** (Lindeløv 2020) via `mcp::mcp` + JAGS:
  ```python
  res = segment_with('rjmcmc', y, n_segments=3, n_iter=3000, n_chains=2)
  ```
